# WTI Producer Hedge Simulator

**Business question:** How much can a crude producer reduce revenue volatility by hedging expected production with WTI futures?

A producer is naturally long physical crude. This notebook tests how short WTI futures can reduce that revenue risk.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.hedge_engine import HedgeAssumptions, prepare_monthly_market_data, compare_hedge_ratios, stress_test


## Download and prepare WTI data

In [ ]:
spot = web.DataReader('DCOILWTICO', 'fred', '2018-01-01')['DCOILWTICO']
raw = yf.download('CL=F', start='2018-01-01', auto_adjust=False, progress=False)
futures = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
market = prepare_monthly_market_data(spot, futures)
market.tail()


## Compare hedge ratios for 100,000 barrels/month

In [ ]:
assumptions = HedgeAssumptions(monthly_production_bbl=100_000)
summary, simulations = compare_hedge_ratios(market, assumptions=assumptions)
summary


## Stress test: 25% crude-price decline with a 75% hedge

In [ ]:
stress_test(spot_price=75, futures_entry=76, spot_shock_pct=-0.25, hedge_ratio=0.75, assumptions=assumptions)
